<h1>QPRU: Qauntum Prototypical Reccurent Unit</h1>

<h1>QGRU: Qauntum Gated Reccurent Unit</h1>

<h1>QLSTM: Qauntum Long-Short Term Memory</h1>

This file contains quantum implementations of QPRU, QGRU, and QLSTM, quantum versions of PRU, GRU, and LSTM using pennylane.

For more information on QPRU you can refer to the paper: https://arxiv.org/abs/2609.04354$0

This jupyter notebook is created for public, shared on github:


The architecture of the QPRU is shown below:
diagram-20260217.svg

Note that c_t is the concatenation of xt and s_t-1.
FC_in and FC_out are the fully connected layers to adjust the dimensionality of the classical data to the VQCs. Though we have used two "FC_in"s and two "FC_out"s, but learning only one pair is enough to handle both the update gate and the output gate.

The architecture of the VQCs used in both recurrent gates is shown in the following picture:
quantum_circuit.svg

In [ ]:
import os, json, time, numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from shutil import rmtree
import pennylane as qml


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
#Save results on Google drive:
SAVE_DIR = "/content/drive/MyDrive/qmodels_results"
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_ROOT = SAVE_DIR
except Exception:
    SAVE_ROOT = "./qmodels_results"
# os.makedirs(SAVE_DIR, exist_ok=True)

progress_file = os.path.join(SAVE_DIR, "progress.json")
results_file = os.path.join(SAVE_DIR, "results_summary.json")


In [ ]:
#params:
n_runs = 10
# Quantum hyperparameters
n_qubits = 4
layers = 2
# Training hyperparameters
epochs = 100
batch_size = 10
learning_rate = 0.01
hidden_size = 5
input_dim = 1
output_dim = 1
window = 3
alpha = 0.7


In [ ]:
#In case if the training crashes because of connection or other reasons, the following functions will help to resume the training:
def load_progress():
    if os.path.exists(progress_file):
        with open(progress_file, "r") as f:
            return json.load(f)
    return {"completed": {}}

def save_progress(progress):
    with open(progress_file, "w") as f:
        json.dump(progress, f, indent=4)

progress = load_progress()


In [ ]:
#Generate synthetic dataset. The dataset is just a simple sin function:
def generate_sine_data_with_window(_from=0, to=8, samples=500, window=20):
    x = np.linspace(_from, to * np.pi, samples)
    y = np.sin(x)
    X_seq, Y_seq = [], []
    for i in range(len(y) - window):
        X_seq.append(y[i: i + window])
        Y_seq.append(y[i + window])
    X_seq = torch.tensor(X_seq, dtype=torch.float32).unsqueeze(-1)
    Y_seq = torch.tensor(Y_seq, dtype=torch.float32).unsqueeze(-1)
    return X_seq, Y_seq


X_all, Y_all = generate_sine_data_with_window(_from=0, to=18, samples=550, window=window)
train_size = int(0.8 * X_all.shape[0])

X_train, X_val = X_all[:train_size], X_all[train_size:]
Y_train, Y_val = Y_all[:train_size], Y_all[train_size:]

train_loader = DataLoader(TensorDataset(X_train, Y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, Y_val), batch_size=batch_size, shuffle=False)

X_train_device, Y_train_device = X_train.to(device), Y_train.to(device)
X_val_device, Y_val_device = X_val.to(device), Y_val.to(device)


In [ ]:
# The VQC circuit:
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev, interface="torch", diff_method="parameter-shift")
def _vqc(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(n_qubits))
    for _ in range(layers):
        for i in range(n_qubits):
            qml.RX(weights[i], wires=i)
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.CNOT(wires=[n_qubits - 1, 0])
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


#BASE MODEL:
class QuantumBase(nn.Module):
    def quantum_circuit(self, inputs, weights):
        return torch.tensor(_vqc(inputs.cpu(), weights.cpu()), dtype=torch.float32)


In [ ]:
#QPRU class:
class QuantumPRU(QuantumBase):
    def __init__(self, input_dim, output_dim, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.fc_in = nn.Linear(input_dim + hidden_size, n_qubits)
        self.weights_update = nn.Parameter(torch.rand(n_qubits))
        self.weights_out = nn.Parameter(torch.rand(n_qubits))
        self.fc_out = nn.Linear(n_qubits, hidden_size)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h_t = torch.zeros((batch_size, self.hidden_size), device=x.device)
        for t in range(seq_len):
            x_t = x[:, t, :]
            combined = torch.cat((h_t, x_t), dim=1)
            x_up = self.fc_in(combined)
            x_out = self.fc_in(combined)
            x_up = torch.stack([self.quantum_circuit(i, self.weights_update) for i in x_up])
            x_out = torch.stack([self.quantum_circuit(i, self.weights_out) for i in x_out])
            x_up = torch.sigmoid(self.fc_out(x_up.to(x.device)))
            x_out = torch.tanh(self.fc_out(x_out.to(x.device)))
            h_t = (1 - x_up) * h_t + x_up * x_out
        return self.fc(h_t)


In [ ]:
#QGRU class:
class QuantumGRU(QuantumBase):
    def __init__(self, input_dim, output_dim, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.fc_in = nn.Linear(input_dim + hidden_size, n_qubits)
        self.weights_reset = nn.Parameter(torch.rand(n_qubits))
        self.weights_update = nn.Parameter(torch.rand(n_qubits))
        self.weights_out = nn.Parameter(torch.rand(n_qubits))
        self.fc_out = nn.Linear(n_qubits, hidden_size)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h_t = torch.zeros((batch_size, self.hidden_size), device=x.device)
        for t in range(seq_len):
            x_t = x[:, t, :]
            combined = torch.cat((h_t, x_t), dim=1)
            x_up = self.fc_in(combined)
            x_re = self.fc_in(combined)
            x_up = torch.stack([self.quantum_circuit(i, self.weights_update) for i in x_up])
            x_re = torch.stack([self.quantum_circuit(i, self.weights_reset) for i in x_re])
            z = torch.sigmoid(self.fc_out(x_up.to(x.device)))
            r = torch.sigmoid(self.fc_out(x_re.to(x.device)))
            combined2 = torch.cat((r * h_t, x_t), dim=1)
            x_tilde = self.fc_in(combined2)
            x_tilde = torch.stack([self.quantum_circuit(i, self.weights_out) for i in x_tilde])
            x_tilde = torch.tanh(self.fc_out(x_tilde.to(x.device)))
            h_t = (1 - z) * x_tilde + z * h_t
        return self.fc(h_t)


In [ ]:
#QLSTM class:
class QuantumLSTM(QuantumBase):
    def __init__(self, input_dim, output_dim, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        self.fc_in = nn.Linear(input_dim + hidden_size, n_qubits)
        self.weights_input = nn.Parameter(torch.rand(n_qubits))
        self.weights_forget = nn.Parameter(torch.rand(n_qubits))
        self.weights_output = nn.Parameter(torch.rand(n_qubits))
        self.weights_cell = nn.Parameter(torch.rand(n_qubits))
        self.fc_out = nn.Linear(n_qubits, hidden_size)
        self.fc = nn.Linear(hidden_size, output_dim)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape
        h_t = torch.zeros((batch_size, self.hidden_size), device=x.device)
        c_t = torch.zeros((batch_size, self.hidden_size), device=x.device)
        for t in range(seq_len):
            x_t = x[:, t, :]
            combined = torch.cat((h_t, x_t), dim=1)
            g = self.fc_in(combined)
            i_v = torch.stack([self.quantum_circuit(i, self.weights_input) for i in g])
            f_v = torch.stack([self.quantum_circuit(i, self.weights_forget) for i in g])
            o_v = torch.stack([self.quantum_circuit(i, self.weights_output) for i in g])
            c_v = torch.stack([self.quantum_circuit(i, self.weights_cell) for i in g])
            i_t = torch.sigmoid(self.fc_out(i_v.to(x.device)))
            f_t = torch.sigmoid(self.fc_out(f_v.to(x.device)))
            o_t = torch.sigmoid(self.fc_out(o_v.to(x.device)))
            g_t = torch.tanh(self.fc_out(c_v.to(x.device)))
            c_t = f_t * c_t + i_t * g_t
            h_t = o_t * torch.tanh(c_t)
        return self.fc(h_t)

In [ ]:
#Train and evaluation functions:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total = 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
        total += loss.item() * xb.size(0)
    return total / len(loader.dataset)

def evaluate(model, loader, loss_fn):
    model.eval()
    total = 0
    preds, trues = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            total += loss_fn(pred, yb).item() * xb.size(0)
            preds.append(pred.cpu())
            trues.append(yb.cpu())
    preds = torch.cat(preds)
    trues = torch.cat(trues)
    return total / len(loader.dataset), preds, trues


In [ ]:
#Loss function and model list:
mae_fn = nn.L1Loss()

models_original = {
      "QPRU": QuantumPRU,
    "QGRU": QuantumGRU,
      "QLSTM": QuantumLSTM,
}


In [ ]:
#Storage for results:
if os.path.exists(results_file):
    with open(results_file, "r") as f:
        results_summary = json.load(f)
else:
    results_summary = {}


In [ ]:
#main training loop and aggregation of results- Note that we run each model ten times to get better statistical understanding of the models:
for name, ModelClass in models_original.items():

    if name not in results_summary:
        results_summary[name] = {"runs": [], "aggregate": {}}

    for run in range(1, n_runs + 1):

        # skip completed runs
        if progress["completed"].get(name, 0) >= run:
            print(f"Skipping {name} run {run} (already completed)")
            continue

        print(f"\n============================================")
        print(f" Training {name} (run {run}/{n_runs})")
        print(f"============================================")

        model = ModelClass(input_dim, output_dim, hidden_size).to(device)
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, alpha=alpha)
        criterion = nn.MSELoss()

        train_losses, val_losses = [], []
        t0 = time.time()

        for epoch in range(1, epochs + 1):
            tr_loss = train_one_epoch(model, train_loader, criterion, optimizer)
            val_loss, _, _ = evaluate(model, val_loader, criterion)
            train_losses.append(tr_loss)
            val_losses.append(val_loss)

            print(f"{name} | Run {run} | Epoch {epoch}/{epochs} "
                  f"| Train {tr_loss:.6f} | Val {val_loss:.6f}")

        t_total = time.time() - t0



        # Predictions for metrics
        t0 = time.time()
        model.eval()
        with torch.no_grad():
            train_pred = model(X_train_device).cpu()
            val_pred = model(X_val_device).cpu()
        t_total_inference= time.time() - t0
        train_true = Y_train.cpu()
        val_true = Y_val.cpu()

        train_mae = mae_fn(train_pred, train_true).item()
        val_mae = mae_fn(val_pred, val_true).item()

        # KS
        ks_train = float(ks_2samp(train_pred.detach().numpy().flatten(),
                                  train_true.detach().numpy().flatten())[0])

        ks_val = float(ks_2samp(val_pred.detach().numpy().flatten(),
                                val_true.detach().numpy().flatten())[0])

        # Save model
        model_path = os.path.join(SAVE_DIR, f"{name}_run{run}.pt")
        torch.save(model.state_dict(), model_path)

        # Save results for this run
        run_stats = {
            "run": run,
            "train_loss_final": float(train_losses[-1]),
            "val_loss_final": float(val_losses[-1]),
            "train_mae": train_mae,
            "val_mae": val_mae,
            "ks_train": ks_train,
            "ks_val": ks_val,
            "train_curve": train_losses,
            "val_curve": val_losses,
            "train_time_sec": t_total,
            "inference_time_sec":t_total_inference,
        }

        results_summary[name]["runs"].append(run_stats)

        # update progress tracker
        progress["completed"][name] = run
        save_progress(progress)

        # save after each run
        with open(results_file, "w") as f:
            json.dump(results_summary, f, indent=4)

    # ============================================================
    # AGGREGATE STATS AFTER ALL RUNS
    # ============================================================
    print(f"\n Computing aggregated results for {name}...")

    runs = results_summary[name]["runs"]

    def mean_std(key):
        vals = [r[key] for r in runs]
        return float(np.mean(vals)), float(np.std(vals))

    agg = {
        "train_loss_mean": mean_std("train_loss_final")[0],
        "train_loss_std": mean_std("train_loss_final")[1],

        "val_loss_mean": mean_std("val_loss_final")[0],
        "val_loss_std": mean_std("val_loss_final")[1],

        "train_mae_mean": mean_std("train_mae")[0],
        "train_mae_std": mean_std("train_mae")[1],

        "val_mae_mean": mean_std("val_mae")[0],
        "val_mae_std": mean_std("val_mae")[1],

        "ks_train_mean": mean_std("ks_train")[0],
        "ks_train_std": mean_std("ks_train")[1],

        "ks_val_mean": mean_std("ks_val")[0],
        "ks_val_std": mean_std("ks_val")[1],
    }

    results_summary[name]["aggregate"] = agg

    # Save after finishing model
    with open(results_file, "w") as f:
        json.dump(results_summary, f, indent=4)


print("\n ALL DONE — results saved in Google Drive!")
print(f"Results JSON: {results_file}")
print(f"Progress JSON: {progress_file}")
